# Pairs Trading — Notebook 3: Walk-Forward Analysis

نوت‌بوک ۲۶ نشون داد که با β ثابت از in-sample، 98% spreads در IS سودده ولی فقط 29% در OOS — **overfit کلاسیک**. ضمناً OOS فقط ۵ ماهه با میانه‌ی ۲ معامله per spread، که آماری ضعیفه.

**این نوت‌بوک هر دو مشکل رو حل می‌کنه:**
1. **β walk-forward refit:** هر `STEP_MONTHS` ماه، α و β از پنجره‌ی متحرک `TRAIN_MONTHS` ماهه refit می‌شن — هیچ‌وقت با اطلاعات آینده.
2. **ADF gate per window:** اگر spread در training window cointegrated نباشه (`adf_p > 0.10`)، اون پنجره رو skip می‌کنیم — کاراکتر دینامیک selection.
3. **OOS بزرگ:** trades فقط در test_window شمرده می‌شن. در نتیجه OOS واقعی ~۲ سال پیوسته (Jan-2024 تا May-2026).

## معماری Walk-Forward
```
        train (12 mo)         test (3 mo)
       |-----------|          |---|
       2023-01 → 2023-12     2024-01 → 2024-03
            train             test
         2023-04 → 2024-03   2024-04 → 2024-06
                     ...
                          2025-03 → 2026-02   2026-03 → 2026-05
```
- **Step = 3 ماه** → ~۹ پنجره برای هر spread
- per-window: refit α/β، ADF gate، backtest فقط روی test_period (با warmup برای rolling z-score)

**ورودی:** shortlist‌های نوت‌بوک ۲۵
**خروجی:** `walkforward_results_{H1,H4}.csv` + `walkforward_trades_{H1,H4}.csv` + `walkforward_betas_{H1,H4}.csv`

In [1]:
from __future__ import annotations
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("ready")

ready


## Configuration

In [2]:
TIMEFRAMES           = ["H1", "H4"]

# Walk-forward window sizes (months)
TRAIN_MONTHS         = 12
TEST_MONTHS          = 3
STEP_MONTHS          = 3
WF_START             = "2023-01-01"   # first day of the first training window
WF_END               = "2026-05-31"   # last day of available data
ADF_GATE_P           = 0.10           # skip window if training-period ADF p > this

# Strategy thresholds (same as notebook 26)
ENTRY_Z              = 2.0
EXIT_Z               = 0.5
STOP_Z               = 4.0
Z_WINDOW_MULT        = 4
Z_WINDOW_MIN         = 100
TIME_STOP_MULT       = 4

# Costs (same as notebook 26)
SPREAD_COST_BPS_PER_LEG_PER_SIDE = 1.0
COMMISSION_BPS_PER_LEG_PER_SIDE  = 0.5
SWAP_BPS_PER_LEG_PER_DAY         = 0.5

DATA_DIR   = PROJECT_ROOT / "notebooks" / "data"
STAT_DIR   = DATA_DIR / "stat_arb"
REAL_TZ    = "Europe/Nicosia"
TF_HOURS   = {"H1": 1, "H4": 4, "D1": 24}

print(f"walk-forward: train={TRAIN_MONTHS}mo  test={TEST_MONTHS}mo  step={STEP_MONTHS}mo")
print(f"range: {WF_START} → {WF_END}")
print(f"ADF gate: skip window if p > {ADF_GATE_P}")

walk-forward: train=12mo  test=3mo  step=3mo
range: 2023-01-01 → 2026-05-31
ADF gate: skip window if p > 0.1


## ۱) Loader + shortlists

In [3]:
def load_pair_h1(symbol: str) -> pd.Series:
    path = DATA_DIR / symbol / "H1" / "ohlcv.csv"
    df = pd.read_csv(path, parse_dates=["time"])
    naive = df["time"].dt.tz_localize(None)
    ts = naive.dt.tz_localize(REAL_TZ, ambiguous="NaT", nonexistent="NaT")
    s = pd.Series(df["close"].values, index=ts, name=symbol)
    return s[s.index.notna()].sort_index()

def to_tf(prices_h1: pd.DataFrame, tf: str) -> pd.DataFrame:
    if tf == "H1":
        return prices_h1
    rule = {"H4": "4h", "D1": "1D"}[tf]
    return prices_h1.resample(rule, label="right", closed="right").last().dropna()

shortlists = {tf: pd.read_csv(STAT_DIR / f"cointegrated_shortlist_{tf}.csv") for tf in TIMEFRAMES}
needed = sorted(set().union(*[set(sl["y"]).union(sl["x"]) for sl in shortlists.values()]))
print(f"symbols needed: {len(needed)}")

h1 = {sym: load_pair_h1(sym) for sym in needed}
all_h1 = pd.concat(h1, axis=1, sort=True).dropna()
prices_by_tf = {tf: to_tf(all_h1, tf) for tf in TIMEFRAMES}
for tf, p in prices_by_tf.items():
    print(f"  [{tf}] bars={len(p)}  range={p.index.min()} → {p.index.max()}")

symbols needed: 28
  [H1] bars=24877  range=2022-05-16 23:00:00+03:00 → 2026-05-15 23:00:00+03:00
  [H4] bars=6356  range=2022-05-17 00:00:00+03:00 → 2026-05-16 00:00:00+03:00


## ۲) Window generator + per-window backtest

In [4]:
def make_windows(start: str, end: str, train_mo: int, test_mo: int, step_mo: int,
                 tz: str = REAL_TZ) -> list[tuple[pd.Timestamp, pd.Timestamp, pd.Timestamp, pd.Timestamp]]:
    end_ts = pd.Timestamp(end, tz=tz)
    windows = []
    train_start = pd.Timestamp(start, tz=tz)
    while True:
        train_end = train_start + pd.DateOffset(months=train_mo)
        test_end  = train_end   + pd.DateOffset(months=test_mo)
        if test_end > end_ts:
            break
        windows.append((train_start, train_end, train_end, test_end))
        train_start += pd.DateOffset(months=step_mo)
    return windows

wins_demo = make_windows(WF_START, WF_END, TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS)
print(f"generated {len(wins_demo)} walk-forward windows")
for i, (ts, te, vs, ve) in enumerate(wins_demo):
    print(f"  #{i+1:>2}  train [{ts.date()} → {te.date()}]  test [{vs.date()} → {ve.date()}]")

generated 9 walk-forward windows
  # 1  train [2023-01-01 → 2024-01-01]  test [2024-01-01 → 2024-04-01]
  # 2  train [2023-04-01 → 2024-04-01]  test [2024-04-01 → 2024-07-01]
  # 3  train [2023-07-01 → 2024-07-01]  test [2024-07-01 → 2024-10-01]
  # 4  train [2023-10-01 → 2024-10-01]  test [2024-10-01 → 2025-01-01]
  # 5  train [2024-01-01 → 2025-01-01]  test [2025-01-01 → 2025-04-01]
  # 6  train [2024-04-01 → 2025-04-01]  test [2025-04-01 → 2025-07-01]
  # 7  train [2024-07-01 → 2025-07-01]  test [2025-07-01 → 2025-10-01]
  # 8  train [2024-10-01 → 2025-10-01]  test [2025-10-01 → 2026-01-01]
  # 9  train [2025-01-01 → 2026-01-01]  test [2026-01-01 → 2026-04-01]


In [5]:
@dataclass
class Trade:
    entry_time: pd.Timestamp
    exit_time:  pd.Timestamp
    side:       int
    entry_spread: float
    exit_spread:  float
    entry_z:    float
    exit_z:     float
    gross_pnl:  float
    cost:       float
    net_pnl:    float
    duration_bars: int
    duration_days: float
    exit_reason:  str
    window_idx:   int
    train_beta:   float


def fit_ols_log(y: np.ndarray, x: np.ndarray) -> tuple[float, float, float]:
    """Return (alpha, beta, adf_p) on log prices."""
    X = np.column_stack([np.ones_like(x), x])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    alpha, beta = float(coef[0]), float(coef[1])
    resid = y - (alpha + beta * x)
    try:
        _, adf_p, *_ = adfuller(resid, regression="n", autolag="AIC")
    except Exception:
        adf_p = 1.0
    return alpha, beta, float(adf_p)


def backtest_window(
    y_sym: str, x_sym: str, alpha: float, beta: float, half_life_bars: float,
    prices: pd.DataFrame, tf: str,
    test_start: pd.Timestamp, test_end: pd.Timestamp,
    window_idx: int,
) -> list[Trade]:
    """Backtest ONLY trades whose entry falls in [test_start, test_end).

    Spread + z-score computed across the full provided prices (which should
    include a warmup chunk *before* test_start so rolling stats are valid by
    the time we reach test_start).
    """
    bar_hours = TF_HOURS[tf]
    z_window  = max(int(round(half_life_bars * Z_WINDOW_MULT)), Z_WINDOW_MIN)
    time_stop = max(int(round(half_life_bars * TIME_STOP_MULT)), Z_WINDOW_MIN)

    py, px = prices[y_sym], prices[x_sym]
    spread = np.log(py) - beta * np.log(px) - alpha
    mu = spread.rolling(z_window, min_periods=z_window).mean()
    sd = spread.rolling(z_window, min_periods=z_window).std()
    z  = (spread - mu) / sd
    valid = z.dropna()
    if valid.empty:
        return []

    legs_per_rt   = 2 * 2
    fixed_cost_bp = legs_per_rt * (SPREAD_COST_BPS_PER_LEG_PER_SIDE + COMMISSION_BPS_PER_LEG_PER_SIDE)
    fixed_cost    = fixed_cost_bp / 1e4

    trades: list[Trade] = []
    position = 0
    entry_t = entry_spread = entry_z_val = None
    entry_idx = -1

    times = valid.index.to_numpy()
    z_vals = valid.to_numpy()
    s_vals = spread.reindex(valid.index).to_numpy()
    in_test = (valid.index >= test_start) & (valid.index < test_end)

    def close_trade(i: int, reason: str) -> None:
        nonlocal position, entry_t, entry_spread, entry_z_val, entry_idx
        exit_t = pd.Timestamp(times[i])
        exit_s = float(s_vals[i])
        exit_z_val = float(z_vals[i])
        gross = position * (exit_s - entry_spread)
        duration_bars = i - entry_idx
        duration_days = duration_bars * bar_hours / 24.0
        swap_cost = 2 * SWAP_BPS_PER_LEG_PER_DAY * duration_days / 1e4
        total_cost = fixed_cost + swap_cost
        trades.append(Trade(
            entry_time=entry_t, exit_time=exit_t, side=position,
            entry_spread=entry_spread, exit_spread=exit_s,
            entry_z=entry_z_val, exit_z=exit_z_val,
            gross_pnl=gross, cost=total_cost, net_pnl=gross - total_cost,
            duration_bars=duration_bars, duration_days=duration_days,
            exit_reason=reason,
            window_idx=window_idx, train_beta=beta,
        ))
        position = 0
        entry_t = entry_spread = entry_z_val = None
        entry_idx = -1

    for i, z_t in enumerate(z_vals):
        if position == 0:
            # Only OPEN trades whose entry timestamp lies inside the test window.
            if not in_test[i]:
                continue
            if z_t > ENTRY_Z:
                position, entry_idx = -1, i
                entry_t = pd.Timestamp(times[i])
                entry_spread, entry_z_val = float(s_vals[i]), float(z_t)
            elif z_t < -ENTRY_Z:
                position, entry_idx = +1, i
                entry_t = pd.Timestamp(times[i])
                entry_spread, entry_z_val = float(s_vals[i]), float(z_t)
        else:
            duration_bars = i - entry_idx
            if abs(z_t) >= STOP_Z and ((position == 1 and z_t < -ENTRY_Z) or (position == -1 and z_t > ENTRY_Z)):
                close_trade(i, "stop_z")
            elif duration_bars >= time_stop:
                close_trade(i, "time_stop")
            elif abs(z_t) <= EXIT_Z:
                close_trade(i, "mean_revert")

    if position != 0:
        close_trade(len(z_vals) - 1, "end_of_window")

    return trades


print("backtest_window defined.")

backtest_window defined.


## ۳) Walk-forward orchestrator per spread

In [6]:
def walk_forward_one_spread(
    y_sym: str, x_sym: str, half_life_bars: float, prices: pd.DataFrame, tf: str,
) -> tuple[list[Trade], list[dict]]:
    """Run walk-forward over all windows for one spread; return (trades, per-window beta logs)."""
    windows = make_windows(WF_START, WF_END, TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS)
    bar_hours = TF_HOURS[tf]
    z_window  = max(int(round(half_life_bars * Z_WINDOW_MULT)), Z_WINDOW_MIN)
    warmup_offset = pd.Timedelta(hours=int(z_window * bar_hours))

    all_trades: list[Trade] = []
    beta_log:   list[dict]  = []

    for w_idx, (ts, te, vs, ve) in enumerate(windows):
        train = prices.loc[ts:te]
        if len(train) < z_window:
            beta_log.append(dict(window=w_idx, test_start=vs, test_end=ve,
                                 alpha=np.nan, beta=np.nan, adf_p=np.nan,
                                 skipped="insufficient_train"))
            continue
        log_y = np.log(train[y_sym].values)
        log_x = np.log(train[x_sym].values)
        alpha, beta, adf_p = fit_ols_log(log_y, log_x)

        if adf_p > ADF_GATE_P or not np.isfinite(beta):
            beta_log.append(dict(window=w_idx, test_start=vs, test_end=ve,
                                 alpha=alpha, beta=beta, adf_p=adf_p,
                                 skipped="adf_gate"))
            continue

        warmup_start = vs - warmup_offset
        window_prices = prices.loc[warmup_start:ve]
        trades = backtest_window(y_sym, x_sym, alpha, beta, half_life_bars,
                                 window_prices, tf, vs, ve, w_idx)
        all_trades.extend(trades)
        beta_log.append(dict(window=w_idx, test_start=vs, test_end=ve,
                             alpha=alpha, beta=beta, adf_p=adf_p,
                             skipped=None, n_trades=len(trades)))

    return all_trades, beta_log


print("walk_forward_one_spread defined.")

walk_forward_one_spread defined.


## ۴) Run for all spreads

In [7]:
def trades_to_df(trades: list[Trade]) -> pd.DataFrame:
    if not trades:
        return pd.DataFrame()
    return pd.DataFrame([t.__dict__ for t in trades])

def annualized_sharpe(pnl: pd.Series, avg_dur_days: float) -> float:
    if pnl.empty or pnl.std() == 0:
        return 0.0
    trades_per_year = 365.0 / max(avg_dur_days, 1.0)
    return float((pnl.mean() / pnl.std()) * np.sqrt(trades_per_year))

all_trades:  dict[str, list[pd.DataFrame]] = {tf: [] for tf in TIMEFRAMES}
all_betas:   dict[str, list[pd.DataFrame]] = {tf: [] for tf in TIMEFRAMES}
all_metrics: dict[str, list[dict]]         = {tf: [] for tf in TIMEFRAMES}

for tf in TIMEFRAMES:
    sl = shortlists[tf]
    prices = prices_by_tf[tf]
    print(f"\n=== [{tf}] walk-forward over {len(sl)} spreads ===")
    for _, row in sl.iterrows():
        y_sym, x_sym = row["y"], row["x"]
        trades, beta_log = walk_forward_one_spread(
            y_sym=y_sym, x_sym=x_sym,
            half_life_bars=row["half_life_bars"],
            prices=prices, tf=tf,
        )
        tdf = trades_to_df(trades)
        if not tdf.empty:
            tdf["y"] = y_sym; tdf["x"] = x_sym
        bdf = pd.DataFrame(beta_log)
        bdf["y"] = y_sym; bdf["x"] = x_sym
        all_trades[tf].append(tdf)
        all_betas[tf].append(bdf)

        if tdf.empty:
            m = dict(y=y_sym, x=x_sym, n_windows_traded=0, n_trades=0,
                     total_pnl=0.0, sharpe=0.0, max_dd=0.0,
                     hit_rate=0.0, avg_dur_days=0.0,
                     beta_mean=np.nan, beta_std=np.nan,
                     skip_rate=float((bdf["skipped"].notna()).mean()))
        else:
            pnl = tdf["net_pnl"]
            eq  = pnl.cumsum()
            dd  = float((eq - eq.cummax()).min())
            traded = bdf[bdf["skipped"].isna()]
            m = dict(
                y=y_sym, x=x_sym,
                n_windows_traded=int((bdf["skipped"].isna()).sum()),
                n_trades=int(len(tdf)),
                total_pnl=float(pnl.sum()),
                sharpe=annualized_sharpe(pnl, tdf["duration_days"].mean()),
                max_dd=dd,
                hit_rate=float((pnl > 0).mean()),
                avg_dur_days=float(tdf["duration_days"].mean()),
                beta_mean=float(traded["beta"].mean()) if len(traded) else np.nan,
                beta_std=float(traded["beta"].std()) if len(traded) > 1 else np.nan,
                skip_rate=float((bdf["skipped"].notna()).mean()),
            )
        all_metrics[tf].append(m)
    print(f"  done. {len(sl)} spreads processed.")


=== [H1] walk-forward over 59 spreads ===
  done. 59 spreads processed.

=== [H4] walk-forward over 47 spreads ===
  done. 47 spreads processed.


## ۵) Summary across all spreads (true OOS)

In [8]:
metrics_by_tf: dict[str, pd.DataFrame] = {}
for tf in TIMEFRAMES:
    m = pd.DataFrame(all_metrics[tf]).sort_values("sharpe", ascending=False).reset_index(drop=True)
    metrics_by_tf[tf] = m
    print(f"\n=== [{tf}] TRUE OOS — top 15 spreads by Sharpe ===")
    cols = ["y", "x", "n_trades", "sharpe", "total_pnl", "hit_rate",
            "max_dd", "avg_dur_days", "beta_mean", "beta_std", "skip_rate"]
    print(m[cols].head(15).to_string(index=False))

    print(f"\n  --- aggregate [{tf}] (TRUE OOS) ---")
    print(f"    median Sharpe:                 {m['sharpe'].median():.2f}")
    print(f"    mean Sharpe:                   {m['sharpe'].mean():.2f}")
    print(f"    median PnL (bps):              {m['total_pnl'].median()*1e4:.1f}")
    print(f"    sum PnL (bps):                 {m['total_pnl'].sum()*1e4:.1f}")
    print(f"    median n_trades per spread:    {m['n_trades'].median():.0f}")
    print(f"    % spreads with positive PnL:   {(m['total_pnl'] > 0).mean()*100:.0f}%")
    print(f"    % spreads with Sharpe > 0.5:   {(m['sharpe'] > 0.5).mean()*100:.0f}%")
    print(f"    % spreads with Sharpe > 1.0:   {(m['sharpe'] > 1.0).mean()*100:.0f}%")
    print(f"    median |beta| volatility (CoV):{(m['beta_std']/m['beta_mean'].abs()).median():.2%}")
    print(f"    avg ADF-gate skip rate:        {m['skip_rate'].mean()*100:.0f}%")


=== [H1] TRUE OOS — top 15 spreads by Sharpe ===
     y      x  n_trades  sharpe  total_pnl  hit_rate  max_dd  avg_dur_days  beta_mean  beta_std  skip_rate
GBPCHF EURNZD         8  6.5235     0.0593    0.8750 -0.0102        8.3490    -0.1619    0.3000     0.2222
EURCHF GBPJPY         8  3.7163     0.0500    0.7500 -0.0038       11.3490     0.0461    0.2391     0.1111
NZDCAD GBPUSD        18  3.7030     0.0675    0.7778 -0.0091        6.5995     0.2801    0.2751     0.0000
NZDCAD EURUSD        16  3.6272     0.0607    0.6875 -0.0056        8.5469     0.3570    0.2548     0.0000
GBPCHF EURUSD        10  3.5298     0.0347    0.7000 -0.0076       10.7167    -0.1711    0.2054     0.2222
NZDCAD USDCAD        14  2.6620     0.0429    0.7143 -0.0097        8.5446    -0.2950    0.1406     0.0000
GBPCHF EURCAD        44  2.0590     0.0576    0.4318 -0.0446        2.5388    -0.0762    0.4347     0.1111
NZDCAD GBPCAD        12  1.9659     0.0442    0.8333 -0.0202       12.2951     0.1266    0.336

## ۶) Equity curve — portfolio-aggregate (همه‌ی spreads با وزن مساوی)

این یک proxy ساده‌ی portfolio‌ـه (equal-weight). نوت‌بوک ۲۸ portfolio construction واقعی + risk-parity می‌گیره.

In [9]:
for tf in TIMEFRAMES:
    tdf_all = pd.concat([t for t in all_trades[tf] if not t.empty], ignore_index=True)
    if tdf_all.empty:
        print(f"[{tf}] no trades")
        continue
    tdf_all = tdf_all.sort_values("exit_time").reset_index(drop=True)
    # Equal-weight across all spreads: each trade contributes 1/N_spreads
    n_spreads = len(shortlists[tf])
    tdf_all["weighted_pnl"] = tdf_all["net_pnl"] / n_spreads
    equity = tdf_all.set_index("exit_time")["weighted_pnl"].cumsum() * 1e4  # in bps

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=equity.index, y=equity.values, mode="lines",
                              name=f"equal-weight portfolio [{tf}]"))
    # Vertical marker at first test_window start (= WF_START + TRAIN_MONTHS)
    oos_start = pd.Timestamp(WF_START, tz=REAL_TZ) + pd.DateOffset(months=TRAIN_MONTHS)
    if not equity.empty:
        ymin, ymax = float(equity.min()), float(equity.max())
        fig.add_trace(go.Scatter(x=[oos_start, oos_start], y=[ymin, ymax],
                                  mode="lines", line=dict(color="red", dash="dash"),
                                  name="first OOS bar"))
    fig.update_layout(title=f"[{tf}] Walk-forward equity (equal-weighted, all spreads, net of costs, bps)",
                      xaxis_title="time", yaxis_title="cumulative PnL (bps)",
                      height=420, hovermode="x unified")
    fig.show()

## ۷) β stability — هر spread، در طول windows

In [10]:
for tf in TIMEFRAMES:
    bdf_all = pd.concat(all_betas[tf], ignore_index=True)
    bdf_all["pair"] = bdf_all["y"] + "~" + bdf_all["x"]
    # Top 5 by walk-forward Sharpe
    top_pairs = metrics_by_tf[tf].head(5).apply(lambda r: f"{r['y']}~{r['x']}", axis=1).tolist()
    fig = go.Figure()
    for p in top_pairs:
        sub = bdf_all[bdf_all["pair"] == p].sort_values("test_start")
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(x=sub["test_start"], y=sub["beta"],
                                  mode="lines+markers", name=p))
    fig.update_layout(title=f"[{tf}] β across walk-forward windows (top 5 spreads)",
                      xaxis_title="test window start", yaxis_title="β (training-window OLS)",
                      height=400, hovermode="x unified")
    fig.show()

## ۸) Save

In [11]:
for tf in TIMEFRAMES:
    m_out = STAT_DIR / f"walkforward_results_{tf}.csv"
    t_out = STAT_DIR / f"walkforward_trades_{tf}.csv"
    b_out = STAT_DIR / f"walkforward_betas_{tf}.csv"
    metrics_by_tf[tf].to_csv(m_out, index=False)
    tdf_concat = [t for t in all_trades[tf] if not t.empty]
    if tdf_concat:
        pd.concat(tdf_concat, ignore_index=True).to_csv(t_out, index=False)
    pd.concat(all_betas[tf], ignore_index=True).to_csv(b_out, index=False)
    print(f"  [{tf}] metrics -> {m_out}")
    print(f"  [{tf}] trades  -> {t_out}")
    print(f"  [{tf}] betas   -> {b_out}")

print("\nnext options (notebook 28):")
print("  A. Portfolio construction with NZD-concentration cap + risk parity sizing")
print("  B. Parameter sensitivity sweep on winners (entry_z, exit_z, z_window)")
print("  C. Live execution skeleton for MT5")

  [H1] metrics -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_results_H1.csv
  [H1] trades  -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_trades_H1.csv
  [H1] betas   -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_betas_H1.csv
  [H4] metrics -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_results_H4.csv
  [H4] trades  -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_trades_H4.csv
  [H4] betas   -> d:\bot\ema-1d trend\ema-h1trend\notebooks\data\stat_arb\walkforward_betas_H4.csv

next options (notebook 28):
  A. Portfolio construction with NZD-concentration cap + risk parity sizing
  B. Parameter sensitivity sweep on winners (entry_z, exit_z, z_window)
  C. Live execution skeleton for MT5
